In [16]:
import datetime
import pytz

edt = pytz.timezone("America/New_York")
today_edt = datetime.datetime.now(edt).date()

print(today_edt)

2026-05-15


In [6]:
from nba_api.stats.endpoints import scoreboardv3
from datetime import timedelta
import pandas as pd

def getUpcomingGames(max_lookahead=3):
    totalGames = []
    query_date = today_edt
    for i in range(max_lookahead):
        query_date = query_date + timedelta(days=i)
        data = scoreboardv3.ScoreboardV3(game_date=query_date)
        games = data.get_data_frames()[1]

        if games.empty:
            continue
        games.drop(columns=["period", "gameClock", "regulationPeriods"], inplace=True)
        games = games[games["ifNecessary"] != True]
        totalGames.append(games)
    df_games = pd.concat(totalGames, ignore_index=True)
    return df_games
        


In [7]:
df_games = getUpcomingGames()
df_games

,gameId,gameCode,gameStatus,gameStatusText,gameTimeUTC,gameEt,seriesGameNumber,gameLabel,gameSubLabel,seriesText,ifNecessary,seriesConference,poRoundDesc,gameSubtype,isNeutral
0,0042500206,20260515/DETCLE,1,7:00 pm ET,2026-05-15T23:00:00Z,2026-05-15T19:00:00Z,Game 6,East Conf. Semifinals,Game 6,CLE leads 3-2,False,East,Conf. Semifinals,,False
1,0042500236,20260515/SASMIN,1,9:30 pm ET,2026-05-16T01:30:00Z,2026-05-15T21:30:00Z,Game 6,West Conf. Semifinals,Game 6,SAS leads 3-2,False,West,Conf. Semifinals,,False


In [43]:
df_games.to_csv("data/upcoming_games.csv", index=False, encoding='utf-8')

In [ ]:
from nba_api.stats.endpoints import scoreboardv3
games = scoreboardv3.ScoreboardV3(game_date=today_edt)


,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,wins,losses,score,seed,inBonus,timeoutsRemaining
0,0042500206,1610612739,Cleveland,Cavaliers,CLE,cavaliers,3,2,0,4,None,0
1,0042500206,1610612765,Detroit,Pistons,DET,pistons,2,3,0,1,None,0
2,0042500236,1610612750,Minnesota,Timberwolves,MIN,timberwolves,2,3,0,6,None,0
3,0042500236,1610612759,San Antonio,Spurs,SAS,spurs,3,2,0,2,None,0


In [24]:
series = games.get_data_frames()[2]
series.drop(columns=["inBonus", "timeoutsRemaining"], inplace=True)
series

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,wins,losses,score,seed
0,0042500206,1610612739,Cleveland,Cavaliers,CLE,cavaliers,3,2,0,4
1,0042500206,1610612765,Detroit,Pistons,DET,pistons,2,3,0,1
2,0042500236,1610612750,Minnesota,Timberwolves,MIN,timberwolves,2,3,0,6
3,0042500236,1610612759,San Antonio,Spurs,SAS,spurs,3,2,0,2


In [25]:
featured_players = games.get_data_frames()[4]
featured_players.drop(columns=["seasonLeadersFlag"], inplace=True)
featured_players

,gameId,teamId,leaderType,personId,name,playerSlug,jerseyNum,position,teamTricode,points,rebounds,assists
0,0042500206,1610612739,home,1628378,Donovan Mitchell,donovan-mitchell,45,G,CLE,26.3,5.3,2.9
1,0042500206,1610612765,away,1630595,Cade Cunningham,cade-cunningham,2,G,DET,30.0,5.5,7.7
2,0042500236,1610612750,home,1630162,Anthony Edwards,anthony-edwards,5,G,MIN,21.3,6.1,2.8
3,0042500236,1610612759,away,1641705,Victor Wembanyama,victor-wembanyama,1,F-C,SAS,20.4,11.2,2.4


In [40]:
import pandas as pd
res = pd.merge(series, featured_players, on='teamId')
res.drop(columns=["gameId_y", "teamTricode_y"], inplace=True)
res.rename(columns={"gameId_x": "gameId", "teamTricode_x": "teamTricode"}, inplace=True)

In [41]:
res

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,wins,losses,score,seed,leaderType,personId,name,playerSlug,jerseyNum,position,points,rebounds,assists
0,0042500206,1610612739,Cleveland,Cavaliers,CLE,cavaliers,3,2,0,4,home,1628378,Donovan Mitchell,donovan-mitchell,45,G,26.3,5.3,2.9
1,0042500206,1610612765,Detroit,Pistons,DET,pistons,2,3,0,1,away,1630595,Cade Cunningham,cade-cunningham,2,G,30.0,5.5,7.7
2,0042500236,1610612750,Minnesota,Timberwolves,MIN,timberwolves,2,3,0,6,home,1630162,Anthony Edwards,anthony-edwards,5,G,21.3,6.1,2.8
3,0042500236,1610612759,San Antonio,Spurs,SAS,spurs,3,2,0,2,away,1641705,Victor Wembanyama,victor-wembanyama,1,F-C,20.4,11.2,2.4


In [42]:
res.to_csv("playoff_series.csv", index=False, encoding='utf-8')

In [35]:
# Update Postgres DB
from sqlalchemy import create_engine
from config import Config

engine = create_engine(Config.SQLALCHEMY_DATABASE_URI)

In [45]:
df_games.columns = df_games.columns.str.lower()

In [46]:
from sqlalchemy import text
df_games.to_sql("upcoming_games", engine, schema="nba_data", if_exists="replace", index=False)
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE nba_data.upcoming_games ADD PRIMARY KEY (gameId);"))
    conn.commit()

In [43]:
res.columns = res.columns.str.lower()

In [34]:
res

,gameid_x,teamid,teamcity,teamname,teamtricode_x,teamslug,wins,losses,score,seed,leadertype,personid,name,playerslug,jerseynum,position,teamtricode_y,points,rebounds,assists
0,0042500206,1610612739,Cleveland,Cavaliers,CLE,cavaliers,3,2,0,4,home,1628378,Donovan Mitchell,donovan-mitchell,45,G,CLE,26.3,5.3,2.9
1,0042500206,1610612765,Detroit,Pistons,DET,pistons,2,3,0,1,away,1630595,Cade Cunningham,cade-cunningham,2,G,DET,30.0,5.5,7.7
2,0042500236,1610612750,Minnesota,Timberwolves,MIN,timberwolves,2,3,0,6,home,1630162,Anthony Edwards,anthony-edwards,5,G,MIN,21.3,6.1,2.8
3,0042500236,1610612759,San Antonio,Spurs,SAS,spurs,3,2,0,2,away,1641705,Victor Wembanyama,victor-wembanyama,1,F-C,SAS,20.4,11.2,2.4


In [45]:
from sqlalchemy import text

res.to_sql("current_series", engine, schema="nba_data", if_exists="replace", index=False)
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE nba_data.current_series ADD PRIMARY KEY (teamid)"))
    conn.commit()